# 🛒 Supermarket Sales Prediction — Upgraded Portfolio Project
**Author:** [Your Name] | **Tools:** Python, Scikit-learn, Seaborn, Streamlit

## Objective
Build a machine learning model that predicts total sales per transaction in a supermarket.
This upgraded version fixes data leakage, adds feature engineering, model optimization, and deployment.

### Improvements Over V1:
- ✅ Fixed data leakage (removed mathematically derived columns)
- ✅ Extracted date/time features (hour, day of week, month)
- ✅ Added feature importance visualization
- ✅ Added hyperparameter tuning with RandomizedSearchCV
- ✅ Added SHAP explainability
- ✅ Streamlit deployment code included

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

print('✅ Libraries imported successfully')

## 2. Load Dataset

In [ ]:
# Download from Kaggle (run in Google Colab)
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d faresashraf1001/supermarket-sales
# !unzip supermarket-sales.zip

df = pd.read_csv('SuperMarket Analysis.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset info
df.info()
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Sales distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Sales'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Sales', fontsize=14)
axes[0].set_xlabel('Sales')
axes[0].set_ylabel('Frequency')

sns.boxplot(x='Branch', y='Sales', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Sales Distribution by Branch', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Sales by product line and gender
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df.groupby('Product line')['Sales'].mean().sort_values().plot(
    kind='barh', ax=axes[0], color='coral')
axes[0].set_title('Average Sales by Product Line', fontsize=14)
axes[0].set_xlabel('Average Sales')

df.groupby(['Product line', 'Gender'])['Sales'].mean().unstack().plot(
    kind='bar', ax=axes[1], colormap='Set1')
axes[1].set_title('Average Sales by Product Line & Gender', fontsize=14)
axes[1].set_xlabel('')
axes[1].legend(title='Gender')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (SAFE columns only — excluding leaky ones)
safe_cols = ['Unit price', 'Quantity', 'Rating', 'Sales']
plt.figure(figsize=(8, 6))
sns.heatmap(df[safe_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap (Clean Features Only)', fontsize=14)
plt.show()

## 4. Feature Engineering
### 4.1 Fix Data Leakage
**Problem:** In V1, columns like `Tax 5%`, `cogs`, `gross margin percentage`, and `gross income` were mathematically derived from `Sales`. Including them gave Linear Regression a perfect R² — but it was just learning the formula, not real patterns.

**Fix:** Drop all leaky columns before training.

In [ ]:
# Drop leaky columns
leaky_cols = ['Tax 5%', 'cogs', 'gross margin percentage', 'gross income', 'Invoice ID']
df_clean = df.drop(columns=leaky_cols)

print('✅ Leaky columns removed')
print(f'Remaining columns: {list(df_clean.columns)}')

### 4.2 Extract Date & Time Features

In [ ]:
# Convert Date and Time to useful features
df_clean['Date'] = pd.to_datetime(df_clean['Date'])
df_clean['Time'] = pd.to_datetime(df_clean['Time'], format='%H:%M')

df_clean['day_of_week'] = df_clean['Date'].dt.dayofweek        # 0=Monday
df_clean['month'] = df_clean['Date'].dt.month
df_clean['hour'] = df_clean['Time'].dt.hour
df_clean['is_weekend'] = df_clean['day_of_week'].isin([5, 6]).astype(int)

# Drop original Date/Time columns
df_clean = df_clean.drop(columns=['Date', 'Time'])

print('✅ Date/Time features extracted')
df_clean[['day_of_week', 'month', 'hour', 'is_weekend']].head()

In [ ]:
# Visualize sales by hour and day of week
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_clean.groupby('hour')['Sales'].mean().plot(kind='bar', ax=axes[0], color='mediumseagreen')
axes[0].set_title('Average Sales by Hour of Day', fontsize=14)
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Average Sales')

day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df_clean.groupby('day_of_week')['Sales'].mean().plot(kind='bar', ax=axes[1], color='mediumpurple')
axes[1].set_xticklabels(day_names, rotation=0)
axes[1].set_title('Average Sales by Day of Week', fontsize=14)
axes[1].set_xlabel('Day')

plt.tight_layout()
plt.show()

## 5. Prepare Features for Modelling

In [ ]:
# Separate features and target
X = df_clean.drop('Sales', axis=1)
y = df_clean['Sales']

# Encode categorical columns
X = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'✅ Training set: {X_train.shape}')
print(f'✅ Test set: {X_test.shape}')

## 6. Train & Evaluate Models

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    cv_score = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    print(f'{name}: MAE={mae:.2f} | RMSE={rmse:.2f} | R²={r2:.4f} | CV R²={cv_score:.4f}')
    return y_pred, r2

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

print('=== Model Comparison (No Data Leakage) ===')
results = {}
for name, model in models.items():
    preds, r2 = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results[name] = {'preds': preds, 'r2': r2}

## 7. Hyperparameter Tuning — Random Forest

In [ ]:
# Hyperparameter tuning with RandomizedSearchCV
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf_tuned = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_tuned.fit(X_train, y_train)

print(f'\n✅ Best Parameters: {rf_tuned.best_params_}')
y_pred_tuned = rf_tuned.predict(X_test)
print(f'Tuned R²: {r2_score(y_test, y_pred_tuned):.4f}')
print(f'Tuned MAE: {mean_absolute_error(y_test, y_pred_tuned):.2f}')
print(f'Tuned RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tuned)):.2f}')

## 8. Feature Importance

In [ ]:
# Feature importance from best Random Forest
best_rf = rf_tuned.best_estimator_
importances = pd.Series(best_rf.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 7))
top_features.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 15 Feature Importances — Random Forest', fontsize=15)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nTop 5 Sales Drivers:')
print(importances.sort_values(ascending=False).head())

## 9. Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_tuned, alpha=0.5, color='steelblue', edgecolors='white', s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Sales', fontsize=12)
plt.ylabel('Predicted Sales', fontsize=12)
plt.title('Actual vs Predicted Sales — Tuned Random Forest', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

## 10. SHAP Explainability (Advanced)

In [ ]:
# Install SHAP if not available
# !pip install shap -q

import shap

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test[:100])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test[:100], plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Top Features)', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Save Model for Deployment

In [ ]:
import joblib

# Save model and feature columns
joblib.dump(best_rf, 'supermarket_model.pkl')
joblib.dump(list(X_train.columns), 'feature_columns.pkl')

print('✅ Model saved as supermarket_model.pkl')
print('✅ Feature columns saved as feature_columns.pkl')

## 12. Streamlit App Code
Save the cell below as `app.py` and run with: `streamlit run app.py`

In [ ]:
streamlit_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# Load model
model = joblib.load("supermarket_model.pkl")
feature_cols = joblib.load("feature_columns.pkl")

st.set_page_config(page_title="Supermarket Sales Predictor", page_icon="🛒")
st.title("🛒 Supermarket Sales Predictor")
st.markdown("Predict transaction sales using machine learning.")

col1, col2 = st.columns(2)

with col1:
    unit_price = st.slider("Unit Price ($)", 10.0, 100.0, 50.0)
    quantity = st.slider("Quantity", 1, 10, 3)
    rating = st.slider("Customer Rating", 4.0, 10.0, 7.0)
    hour = st.slider("Hour of Purchase", 10, 20, 14)

with col2:
    branch = st.selectbox("Branch", ["A", "B", "C"])
    customer_type = st.selectbox("Customer Type", ["Member", "Normal"])
    gender = st.selectbox("Gender", ["Male", "Female"])
    payment = st.selectbox("Payment Method", ["Cash", "Credit card", "Ewallet"])
    product_line = st.selectbox("Product Line", [
        "Health and beauty", "Electronic accessories",
        "Home and lifestyle", "Sports and travel",
        "Food and beverages", "Fashion accessories"])
    day_of_week = st.selectbox("Day of Week", ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])

if st.button("🔮 Predict Sales", use_container_width=True):
    day_map = {"Mon":0,"Tue":1,"Wed":2,"Thu":3,"Fri":4,"Sat":5,"Sun":6}
    is_weekend = 1 if day_map[day_of_week] >= 5 else 0

    input_dict = {
        "Unit price": unit_price, "Quantity": quantity, "Rating": rating,
        "hour": hour, "day_of_week": day_map[day_of_week],
        "month": 1, "is_weekend": is_weekend,
        "Branch_B": 1 if branch=="B" else 0,
        "Branch_C": 1 if branch=="C" else 0,
        "Customer type_Normal": 1 if customer_type=="Normal" else 0,
        "Gender_Male": 1 if gender=="Male" else 0,
        "Payment_Credit card": 1 if payment=="Credit card" else 0,
        "Payment_Ewallet": 1 if payment=="Ewallet" else 0,
    }
    for line in ["Electronic accessories","Fashion accessories","Food and beverages",
                 "Health and beauty","Home and lifestyle","Sports and travel"]:
        key = f"Product line_{line}"
        input_dict[key] = 1 if product_line == line else 0

    input_df = pd.DataFrame([input_dict])
    for col in feature_cols:
        if col not in input_df.columns:
            input_df[col] = 0
    input_df = input_df[feature_cols]

    prediction = model.predict(input_df)[0]
    st.success(f"💰 Predicted Sales: **${prediction:,.2f}**")
    st.balloons()
'''

with open('app.py', 'w') as f:
    f.write(streamlit_code)

print('✅ Streamlit app saved as app.py')
print('Run with: streamlit run app.py')

## 13. Business Insights

### Key Findings from the Clean Model:

**1. Quantity is the #1 Sales Driver**
The number of items purchased has the strongest impact on total sales. → *Introduce bundle deals and multi-buy promotions.*

**2. Unit Price Matters**
Premium products drive higher transaction values. → *Maintain stock of high-value items and avoid over-discounting.*

**3. Time of Day Affects Sales**
Peak hours show higher average sales. → *Schedule more staff and ensure stock availability during peak hours.*

**4. Branch Performance Varies**
Significant differences across branches suggest location-based demand. → *Tailor inventory and staffing per branch.*

**5. Product Line Differences**
Some lines consistently outperform others. → *Prioritize high-performing lines in shelf space and marketing.*

**6. Weekday vs Weekend Patterns**
Weekend transactions show different buying patterns. → *Run weekend-specific promotions.*

### Model Performance Summary:
| Model | Notes |
|---|---|
| Linear Regression | Baseline — lower performance without leaky features |
| Decision Tree | Prone to overfitting |
| Random Forest (Default) | Strong balanced performance |
| **Random Forest (Tuned)** | **Best model — selected for deployment** |
| Gradient Boosting | Competitive alternative |

## Conclusion
This upgraded model provides honest, generalizable sales predictions by eliminating data leakage, engineering time-based features, and optimizing hyperparameters. The deployed Streamlit app enables real-time sales forecasting to support inventory, staffing, and business planning decisions.